# SelfSNE quick demo

Minimal Self-Supervised Noise Embedding training on synthetic data.

In [1]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
from selfsne import SelfSNE
from selfsne.losses import LikelihoodRatioEstimator

/home/jake/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pl.seed_everything(0)
t = torch.linspace(0, 4*torch.pi, 256)
x = torch.stack([t.sin(), t.cos(), (2*t).sin(), (2*t).cos()], dim=1)
ds = TensorDataset(x[:-1], x[1:])
loader = DataLoader(ds, batch_size=64)

Seed set to 0


In [3]:
model = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="kld", baseline="batch"),
    target_encoder=torch.nn.Sequential(
        torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2)
    )
)

trainer = pl.Trainer(max_epochs=3, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer.fit(model, loader)
import numpy as np
emb = trainer.predict(model, loader)
embeddings = np.concatenate([b["target_embedding"] for b in emb])
print("Embeddings shape:", embeddings.shape)

GPU available: True (cuda), used: False


TPU available: False, using: 0 TPU cores


/home/jake/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/home/jake/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss             │ LikelihoodRatioEstimator │      0 │ train │     0 │
│ 1 │ target_encoder   │ Sequential               │    226 │ train │     0 │
│ 2 │ target_tokenizer │ Identity                 │      0 │ train │     0 │
└───┴──────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 226                                                                                              
Non-trainable params: 0                                                                                            
Total params: 226                                                                                                  
Total estimated model params size (MB): 0.001                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/jake/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:497: UserWarning: Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0
- 8.6 which supports hardware CC >=8.6,<9.0
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.15.0+cu132 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.15.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/jake/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:497: UserWarning: Found GPU1 NVIDIA GeForce GTX 1080 Ti which is of compute capability (

`Trainer.fit` stopped: `max_epochs=3` reached.


Embeddings shape: (255, 2)


/home/jake/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


In [4]:
# SelfSNE MI lower bound via likelihood ratio estimation
# LikelihoodRatioEstimator learns f = log p(x,y) - log p(x)p(y)
# Valid DV/MINE-style lower bound (Poole et al. 2019):
#   I(X;Y) >= E_pos[f] - log E_neg[e^f] = trained attr + KLD loss value
import torch
import numpy as np
pos_means, neg_exp_means = [], []
for xb, yb in loader:
    with torch.no_grad():
        ctx_emb, tgt_emb, _, _ = model.get_embeddings(xb, yb, None)
        logits = model.loss.logits(ctx_emb, tgt_emb)
        pos_means.append(logits['pos_logits'].mean().item())
        neg_exp_means.append(torch.exp(logits['neg_logits']).mean().item())
mi_lb = float(np.mean(pos_means) - np.log(np.mean(neg_exp_means)))
print("SelfSNE MI lower bound estimate:", mi_lb)

SelfSNE MI lower bound estimate: 0.05985661968591671


## MI estimation test with known Gaussian MI

Generate bivariate Gaussian with known correlation rho. Train SelfSNE with
`kld` + `batch` baseline — the DV/MINE configuration of Poole et al. (2019):
the `log E_neg[e^k]` repulsion term exactly cancels the batch baseline, leaving
the valid lower bound `I(X;Y) >= E_pos[k] - log E_neg[e^k]`, reported directly by
the training loss. Compare it to the analytic `I = -0.5 log(1 - rho^2)`.

In [5]:
import numpy as np
from torch.utils.data import DataLoader
rho = 0.7
n = 20000
mean = [0,0]
cov = [[1, rho],[rho,1]]
xy = np.random.multivariate_normal(mean, cov, size=n)
x = torch.from_numpy(xy[:,:1].astype(np.float32))
y = torch.from_numpy(xy[:,1:].astype(np.float32))
ds = TensorDataset(x, y)
loader_gauss = DataLoader(ds, batch_size=512, shuffle=True)

In [6]:
true_mi = -0.5 * np.log(1 - rho**2)
print("True Gaussian MI:", true_mi)

True Gaussian MI: 0.3366722766318828


In [7]:
from selfsne.nn.norm import InputNorm1d
pl.seed_everything(1)
encoder = torch.nn.Sequential(
    InputNorm1d(1),
    torch.nn.Linear(1, 128),
    torch.nn.SELU(),
    torch.nn.Linear(128, 64),
    torch.nn.SELU(),
    torch.nn.Linear(64, 2)
)
# DV/MINE: kld + batch baseline -> log E_neg[e^k] == 0, bound = E_pos[k]
model_g = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="kld", baseline="batch"),
    target_encoder=encoder,
    learning_rate=0.003,
    lr_scheduler=True,
    lr_warmup_steps=500,
    lr_target_steps=2000,
    lr_cosine_steps=2000
)
trainer_g = pl.Trainer(max_epochs=50, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer_g.fit(model_g, loader_gauss)

Seed set to 1


GPU available: True (cuda), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss             │ LikelihoodRatioEstimator │      0 │ train │     0 │
│ 1 │ target_encoder   │ Sequential               │  8.6 K │ train │     0 │
│ 2 │ target_tokenizer │ Identity                 │      0 │ train │     0 │
└───┴──────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 8.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.6 K                                                                                                
Total estimated model params size (MB): 0.035                                                                      
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.


In [8]:
model_g.eval()
# Evaluation: f(x,y) = kernel similarity (batch baseline cancels in the bound)
# I(X;Y) >= E_pos[f] - log E_neg[e^f]; with the batch baseline the loss value
# is exactly -(E_pos[k] - log E_neg[e^k]), so both agree.
pos_means, neg_exp_means = [], []
for xb, yb in loader_gauss:
    with torch.no_grad():
        ctx_emb, tgt_emb, _, _ = model_g.get_embeddings(xb, yb, None)
        logits = model_g.loss.logits(ctx_emb, tgt_emb)
        pos_means.append(logits['pos_logits'].mean().item())
        neg_exp_means.append(torch.exp(logits['neg_logits']).mean().item())
mi_lb_g = float(np.mean(pos_means) - np.log(np.mean(neg_exp_means)))
print("E_pos[f]  =", float(np.mean(pos_means)))
print("E_neg[e^f] =", float(np.mean(neg_exp_means)))
print("DV/MINE bound from evaluations:", mi_lb_g)
print("Final trained KLD loss (attr - rep):", trainer_g.callback_metrics)
print("True Gaussian MI:", true_mi)

E_pos[f]  = 0.26433482095599176
E_neg[e^f] = 1.0000000089406966
DV/MINE bound from evaluations: 0.2643348120152952
Final trained KLD loss (attr - rep): {'loss': tensor(-0.3473), 'lr_scale': tensor(1.)}
True Gaussian MI: 0.3366722766318828


In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.figure()
plt.bar(["True MI", "SelfSNE LB"], [true_mi, mi_lb_g])
plt.ylabel("MI")
plt.title("MI estimation: Gaussian ground truth vs SelfSNE lower bound")
plt.show()